# Transformer 块

有了注意力还不够，需要把它包装成可重复堆叠的完整构建块——
这就是 Transformer 块。

GPT-2 Small 堆 12 块，GPT-3 堆 96 块，LLaMA 70B 堆 80 块。
现代语言模型都是重复同一模块，块本身不变，只是数量不同。

每块有两部分：注意力让每个词与其他词交互；
前馈网络让每个词独自消化刚听到的信息。
两部分之间做归一化，并加跳跃连接让原始信号直通。

跳跃连接是深层网络的关键：没有它梯度会消失，
超过几层就训不动；有了它梯度有高速公路回到第一层，
所以能堆 12、96 甚至 100 层并持续学习。

本 notebook 用 RMSNorm 归一化、SwiGLU 前馈，
构建完整 Transformer 块（LLaMA、Mistral 的现代选择），
并纳入上一章的注意力模块以保持 notebook 自包含。

## 导入

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 旋转位置嵌入（RoPE）

来自第 4 章，注意力模块需要它。

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## 因果掩码

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## 多头注意力

来自第 5 章，带 RoPE 与因果掩码的完整注意力模块。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## RMSNorm

均方根归一化，比 LayerNorm 更简单更快：
用向量均方根做除法，无需去均值。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

## SwiGLU

门控前馈网络，大规模下优于 ReLU 或 GELU：
一路产生 value，一路产生 gate 控制通过量。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

## Transformer 块

组装全部组件：两个子层，各带 pre-norm 与残差连接。
注意力在 token 间混合信息；SwiGLU 独立处理每个 token。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## 测试该块

按 GPT-2 规模创建一个 Transformer 块并运行。

In [ ]:
d_model = 768
num_heads = 12
seq_len = 64
batch_size = 4

block = TransformerBlock(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len, x.device)

output = block(x, mask)

print(f"输入形状:  {x.shape}")
print(f"输出形状: {output.shape}")
print(f"形状一致: {x.shape == output.shape}")
print()

diff = (output - x).abs().mean().item()
print(f"块处理后平均变化: {diff:.4f}")
print(f"块改变了输入（确实做了计算）")
print()

params = sum(p.numel() for p in block.parameters())
print(f"每块参数量: {params:,}")
print(f"12 层 GPT-2 Small 共: {params * 12:,} 参数（所有块）")

## 残差连接的作用

将注意力与 FFN 输出置零，
输入应原样通过，得益于跳跃连接。

In [ ]:
block.eval()
with torch.no_grad():
    for p in block.attention.parameters():
        p.zero_()
    for p in block.ffn.parameters():
        p.zero_()

x = torch.randn(1, 4, d_model)
mask = create_causal_mask(4, x.device)
output = block(x, mask)

diff = (output - x).abs().max().item()
print(f"权重为零时最大差异: {diff:.10f}")
print(f"输入原样通过（残差连接有效）")